## Step 5 - Build the FAISS index (+ id_map)

Stacks the per-frame vectors into one inner-product index and writes the aligned
`id_map`. Works for any backbone from Step 4 - just point `FEATURES_DIR` at that
model's folder and set `MODEL_KEY`.

```
<MODEL_KEY>.index   -> backend/data/indexes/<MODEL_KEY>.index
id_map.json         -> backend/data/indexes/id_map.json   (same for every model)
```

`frame_id = <video_name>_<idx>` matches the keys in `frames_meta.json` (Step 2).


In [ ]:
!pip -q install faiss-cpu

In [ ]:
import os, glob, json, faiss
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [ ]:
MODEL_KEY    = 'siglip'                                          # match Step 4
FEATURES_DIR = f'/kaggle/input/aic-clip-features/clip-features/{MODEL_KEY}'
INDEX_PATH   = f'./{MODEL_KEY}.index'
ID_MAP_PATH  = './id_map.json'

In [ ]:
npy_paths = sorted(glob.glob(os.path.join(FEATURES_DIR, '**', '*.npy'), recursive=True))
print('vectors:', len(npy_paths))
assert npy_paths, 'No .npy features found - check FEATURES_DIR / MODEL_KEY'

ids, vectors = [], []
for p in tqdm(npy_paths):
    path = Path(p)
    video_name = path.parent.name          # L21_V001 (thư mục cha trực tiếp của file)
    idx = int(path.stem)                    # 000080 -> 80
    ids.append(f'{video_name}_{idx:06d}')   # frame_id, matches frames_meta.json
    vectors.append(np.load(p).astype(np.float32).reshape(-1))

matrix = np.vstack(vectors)
faiss.normalize_L2(matrix)                  # unit norm -> IP == cosine
print('matrix:', matrix.shape)

In [ ]:
index = faiss.IndexFlatIP(matrix.shape[1])
index.add(matrix)
faiss.write_index(index, INDEX_PATH)
with open(ID_MAP_PATH, 'w', encoding='utf-8') as f:
    json.dump(ids, f, ensure_ascii=False)
print('index vectors:', index.ntotal)
print('wrote', INDEX_PATH, 'and', ID_MAP_PATH)

### Deploy

```text
<MODEL_KEY>.index -> backend/data/indexes/<MODEL_KEY>.index
id_map.json       -> backend/data/indexes/id_map.json
frames_meta.json  -> backend/data/metadata/frames_meta.json   (Step 2)
Keyframes/*       -> backend/media/frames/                     (Step 3)
```

Query the API with `model=<MODEL_KEY>`, and make sure the backend encodes text
queries with the **same** model (and that `_DIMS` lists its dim).
